# Kaggle ↔ Colab 一键桥(全自动版)

**只需要 3 步,没有别的:**
1. 点 **「代码执行程序 → 全部运行」**(或左边每格的 ▶ 从上到下);
2. 中途弹出 `Kaggle API Token` 输入框 → **粘贴你的密钥** → 回车;
3. 等 10~20 分钟,最后一格显示 `✅ 报告已回传` 即完成。

> 本笔记本自带执行链:探针 → 往 Kaggle 推 2×T4 GPU kernel → 取回结果 → 自动回传给助手。
> **不用**挂载 Drive、**不用** Secrets 面板、**不用**选账号。
> 密钥只在内存中使用,不写入任何文件,不会打印。

In [ ]:
%pip install -q "kaggle==2.2.5"
import os, platform, subprocess, sys
print("python:", sys.version.split()[0], "|", platform.platform())
try:
    import torch
    print("torch:", torch.__version__, "| cuda:", torch.cuda.is_available(),
          "| devices:", [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])
except Exception as e:
    print("torch:", e)
smi = subprocess.run(["nvidia-smi", "--query-gpu=index,name,memory.total", "--format=csv,noheader"],
                     capture_output=True, text=True)
print(smi.stdout or "[no nvidia-smi — 点菜单:代码执行程序 → 更改运行时类型 → T4 GPU]")

In [ ]:
# ---- 粘贴密钥(仅内存,不落盘、不打印) ----
import getpass, os
_raw = getpass.getpass("Kaggle API Token / Key(粘贴后回车,不回显): ").strip()
CRED = {}
if ":" in _raw and not _raw.startswith(("KGAT_", "KGRT_", "sk-")):
    u, k = _raw.split(":", 1)
    CRED = {"KAGGLE_USERNAME": u.strip(), "KAGGLE_KEY": k.strip()}
elif _raw:
    CRED = {"KAGGLE_API_TOKEN": _raw}
else:
    raise ValueError("需要密钥才能继续(kaggle.com/settings/api → Create New API Token)")
del _raw
print("credential loaded:", "token" if "KAGGLE_API_TOKEN" in CRED else "user+key")

In [ ]:
# ---- 探针:验证环境与 Kaggle 登录(只读) ----
import json, subprocess, sys, time
REPORT = {"started_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())}

def sh(args, timeout=120):
    return subprocess.run(args, capture_output=True, text=True, timeout=timeout)

probe = {"probe": "colab-kaggle-v2", "python": sys.version.split()[0],
         "kaggle_secret_source": "KGAT" if "KAGGLE_API_TOKEN" in CRED else "legacy"}
smi = sh(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], 30)
probe["gpu"] = smi.stdout.strip() or "no nvidia-smi"
if "KAGGLE_API_TOKEN" in CRED:
    os.makedirs(os.path.expanduser("~/.kaggle"), exist_ok=True)
    p = os.path.expanduser("~/.kaggle/access_token")
    with open(p, "w") as f:
        f.write(CRED["KAGGLE_API_TOKEN"] + "\n")
    os.chmod(p, 0o600)
env = os.environ.copy(); env.update(CRED)
auth = sh([sys.executable, "-m", "kaggle", "kernels", "list", "--mine", "--csv", "--page-size", "1"], 90)
probe["kaggle_auth_probe_exit"] = auth.returncode
probe["kaggle_authenticated"] = auth.returncode == 0
print("COLAB_EXECUTE_OK", flush=True)
print("KAGGLE_AUTH_OK" if auth.returncode == 0 else "KAGGLE_AUTH_FAILED:\n" + (auth.stderr or auth.stdout)[-400:], flush=True)
REPORT["probe"] = probe

In [ ]:
# ---- 冒烟:推 Kaggle 2xT4 GPU kernel -> 轮询 -> 取回结果 ----
import json, os, subprocess, sys, time
from pathlib import Path

def sh(args, timeout=120):
    return subprocess.run(args, capture_output=True, text=True, timeout=timeout)

env = os.environ.copy(); env.update(CRED)
if "KAGGLE_API_TOKEN" in CRED:
    os.makedirs(os.path.expanduser("~/.kaggle"), exist_ok=True)
    p = os.path.expanduser("~/.kaggle/access_token")
    with open(p, "w") as f:
        f.write(CRED["KAGGLE_API_TOKEN"] + "\n")
    os.chmod(p, 0o600)

uname = ""
try:
    from kaggle.api.kaggle_api_extended import KaggleApi
    api = KaggleApi(); api.authenticate()
    uname = (api.config_values.get("username") or "").strip()
except Exception as exc:
    print(f"username via api: {type(exc).__name__}", flush=True)
if not uname:
    r = sh([sys.executable, "-m", "kaggle", "kernels", "list", "--mine", "--csv", "--page-size", "1"], 90)
    for line in (r.stdout or "").splitlines()[1:]:
        if "/" in line:
            uname = line.split(",")[0].split("/")[0].strip(); break

smoke = {"job": "kaggle-t4-smoke", "kaggle_user": uname or "unknown"}
if not uname:
    smoke["status"] = "failed"; smoke["reason"] = "cannot resolve kaggle username"
    REPORT["smoke"] = smoke
    print(json.dumps(smoke, indent=2))
else:
    slug = "arena-t4-smoke-20261113"
    stage = Path("/tmp/arena_kernel"); stage.mkdir(exist_ok=True)
    KERNEL = 'import json, subprocess, time\nimport torch\nout = {"ts": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),\n       "torch": torch.__version__, "cuda": torch.cuda.is_available(),\n       "devices": [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())]}\nif out["cuda"]:\n    x = torch.randn(2048, 2048, device="cuda")\n    out["matmul_mean"] = float((x @ x).mean().item())\nsmi = subprocess.run(["nvidia-smi", "--query-gpu=index,name,memory.total",\n                      "--format=csv,noheader"], capture_output=True, text=True)\nout["nvidia_smi"] = smi.stdout.strip()\nwith open("/kaggle/working/smoke_result.json", "w") as f:\n    json.dump(out, f, indent=2)\nprint(json.dumps(out))'
    (stage / "main.py").write_text(KERNEL)
    meta = {"id": f"{uname}/{slug}", "title": "arena t4 smoke 20261113",
            "code_file": "main.py", "language": "python", "kernel_type": "script",
            "is_private": "true", "enable_gpu": "true", "enable_internet": "true"}
    (stage / "kernel-metadata.json").write_text(json.dumps(meta, indent=2))
    r = sh([sys.executable, "-m", "kaggle", "kernels", "push", "-p", str(stage)], 120)
    smoke["push_exit"] = r.returncode
    if r.returncode != 0:
        smoke["status"] = "failed"; smoke["push_err"] = ((r.stderr or r.stdout or "")[-500:])
    else:
        ref = f"{uname}/{slug}"
        smoke["kernel_url"] = f"https://www.kaggle.com/code/{ref}"
        print("pushed", ref, flush=True)
        deadline = time.time() + 1500
        final = "timeout"
        while time.time() < deadline:
            time.sleep(30)
            r = sh([sys.executable, "-m", "kaggle", "kernels", "status", ref], 60)
            txt = ((r.stdout or "") + (r.stderr or "")).lower()
            print("status:", txt.strip()[:100], flush=True)
            if "complete" in txt: final = "completed"; break
            if "error" in txt or "cancel" in txt: final = "failed"; break
        smoke["kernel_status"] = final
        outdir = Path("/tmp/arena_kout")
        r = sh([sys.executable, "-m", "kaggle", "kernels", "output", ref, "-p", str(outdir), "-o"], 300)
        smoke["output_exit"] = r.returncode
        smoke["outputs"] = {}
        for f in sorted(outdir.rglob("*")):
            if f.is_file() and f.stat().st_size <= 2 * 1024 * 1024:
                smoke["outputs"][f.name] = f.read_text("utf-8", "replace")[:8000]
        smoke["status"] = final
    REPORT["smoke"] = smoke
print(json.dumps(REPORT.get("smoke", {}), ensure_ascii=False, indent=2)[:2000])

In [ ]:
# ---- 回传报告给助手(一次性预授权通道;失败则把 JSON 贴回会话) ----
import json, time, urllib.request
REPORT["finished_utc"] = time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())
blob = json.dumps(REPORT, ensure_ascii=False, indent=2)
with open("/content/arena_colab_report.json", "w", encoding="utf-8") as f:
    f.write(blob)
try:
    req = urllib.request.Request(
        'https://www.googleapis.com/upload/drive/v3/files?uploadType=resumable&supportsAllDrives=True&upload_id=AFy3sg3sfzvU501-p43vr0uITA_f4EEJRohtWQIqAUGleJ6bZJ67AL5L8Hx-6Y6kisPfVYM1I0_tLu-yjI8qjP_-ohJxw7eKfmQpgQgHFh9EQ-Y&session_crd=AHSoBRXYBgxVnm1GEgApQk8Ui15haEmKWSf0jLBo2GQLjtahUPWnVEdb5zouBtrhbN0kXsH2H16VxosP4Xo&upload_protocol=resumable',
        data=blob.encode("utf-8"), method="PUT",
        headers={"Content-Type": "application/json", "Content-Length": str(len(blob)),
                 "X-Goog-Upload-Offset": "0", "X-Goog-Upload-Command": "upload, finalize",
                 "X-Goog-Upload-Total-Size": str(len(blob)),
                 "Content-Range": f"bytes 0-{max(len(blob) - 1, 0)}/{len(blob)}"})
    with urllib.request.urlopen(req, timeout=120) as resp:
        ok = resp.headers.get("X-Goog-Upload-Status") == "final"
    print("✅ 报告已回传,助手会自动验收!" if ok else "⚠️ 上传未确认,请把 /content/arena_colab_report.json 内容贴回会话")
except Exception as exc:
    print(f"⚠️ 回传失败({type(exc).__name__}),请把下面 JSON 贴回会话:\n", blob[:4000])
print("\n--- REPORT ---\n" + blob[:3000])

---

**故障自查(极少用到):**
- `KAGGLE_AUTH_FAILED` → 密钥不对/过期,去 [kaggle.com/settings/api](https://www.kaggle.com/settings/api) 重新 Create New API Token,重跑全部;
- `[no nvidia-smi]` → 菜单「代码执行程序 → 更改运行时类型」选 **T4**,重跑;
- kernel 状态一直 `running` → Kaggle 排队中,耐心等(最长 25 分钟);
- 想再跑一次:直接「全部运行」重来(冒烟 kernel 会自动重建)。
